# Make Probdiffeq's solvers reverse-mode differentiable

Use [Equinox's](https://docs.kidger.site/equinox/)
bounded while loop to enable reverse-mode differentiation of adaptive IVP solvers.

In [1]:
"""Use Equinox's while loop to compute gradients of `simulate_terminal_values`."""

import equinox
import jax
import jax.numpy as jnp

from probdiffeq import ivpsolve, probdiffeq, taylor

# Fail this notebook on NaN detection (to catch those in the CI)
jax.config.update("jax_debug_nans", True)


def solution_routine(while_loop):
    """Construct a parameter-to-solution function and an initial value."""

    @jax.jit
    def vf(y, /, *, t):  # noqa: ARG001
        """Evaluate the vector field."""
        return 0.5 * y * (1 - y)

    t0, t1 = 0.0, 1.0
    u0 = jnp.asarray([0.1])

    tcoeffs = taylor.odejet_padded_scan(lambda y: vf(y, t=t0), (u0,), num=1)
    init, ssm = probdiffeq.ssm_taylor(tcoeffs, ssm_fact="isotropic")
    iwp = probdiffeq.prior_wiener_integrated(ssm=ssm)
    ts0 = probdiffeq.constraint_ode_ts0(vf, ssm=ssm)

    strategy = probdiffeq.strategy_smoother_fixedpoint(ssm=ssm)
    solver = probdiffeq.solver(strategy=strategy, prior=iwp, constraint=ts0, ssm=ssm)
    error = probdiffeq.error_residual_std(constraint=ts0, prior=iwp, ssm=ssm)
    solve_adaptive = ivpsolve.solve_adaptive_terminal_values(
        solver=solver, error=error, while_loop=while_loop
    )

    def simulate(init_val):
        """Evaluate the parameter-to-solution function."""
        sol = solve_adaptive(init_val, t0=t0, t1=t1, atol=1e-3, rtol=1e-3)

        # Any scalar function of the IVP solution would do
        # Try the log-marginal-likelihood losses (see the other tutorials).
        return jnp.dot(sol.u.mean[0], sol.u.mean[0])

    return simulate, init

This is the default behaviour.

In [2]:
solve, x = solution_routine(jax.lax.while_loop)

try:
    solution, gradient = jax.jit(jax.value_and_grad(solve))(x)
except ValueError as err:
    print(f"Caught error:\n\t {err}")

Caught error:
	 Reverse-mode differentiation does not work for lax.while_loop or lax.fori_loop with dynamic start/stop values. Try using lax.scan, or using fori_loop with static start/stop.


This while-loop makes the solver differentiable

In [3]:
def while_loop_func(*a, **kw):
    """Evaluate a bounded while loop."""
    return equinox.internal.while_loop(*a, **kw, kind="bounded", max_steps=100)


solve, x = solution_routine(while_loop=while_loop_func)

# Compute gradients
solution, gradient = jax.jit(jax.value_and_grad(solve))(x)

print(solution)
print(gradient)

0.023921324
TaylorCoeffTarget(marginals=IsotropicNormal(mean=[[0.44211525]
 [0.01909959]], cholesky=[[0. 0.]
 [0. 0.]], treedef=PyTreeDef([*, *])))
